# ODS fingerprints: noticing a row changed after its load

ODS is append-only: a parser change appends rows under a new version, and nothing updates a row in place (plan, Point-in-time rules; F3). The time machine leans on this. A replay at a past `T` rebuilds DWD and DWS from ODS, so a changed ODS row would change a past forecast without anyone seeing it. dbt never writes ODS, and BigQuery has no insert-only role, so no rule over dbt's SQL can stop an `UPDATE` from elsewhere. It can only be noticed: each file's rows get a fingerprint at load, and a check finds any file whose rows no longer match (rules README; plan, Warehouse tests).

A recheck that reads all of ODS after every load would cost what ODS weighs, several times a day. This notebook asks what is needed to recheck cheaply.

| Question | Where |
| --- | --- |
| What does the loader record per file, and does ODS agree with it today? | Section 1 |
| Which fingerprint, and what does it cost per file and per table? | Section 2 |
| After a change in place, what shows it without reading the rows? | Section 3 |

## 0. Setup

**Step.** `sql()` runs a query against ODS in dev and prints what it scanned. `FP` is the fingerprint the spike starts from, explained in section 2: each row as JSON without its nulls, hashed with `FARM_FINGERPRINT`, the hashes summed exactly as `BIGNUMERIC`.

In [1]:
import os
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from google.cloud import bigquery

from jtf.parse import PARSERS
from jtf.parse.bigquery import BigQueryWarehouse
from jtf.parse.load import MANIFEST_COLUMNS

pd.set_option("display.width", 160, "display.max_columns", 30, "display.max_colwidth", 70)


def setting(name: str) -> str:
    if value := os.environ.get(name):
        return value
    lines = Path("../.env").read_text().splitlines()
    env = dict(line.split("=", 1) for line in lines if "=" in line and not line.startswith("#"))
    return env[name]


PROJECT = setting("JTF_BIGQUERY_PROJECT")
LOCATION = setting("JTF_BIGQUERY_LOCATION")
ODS = f"{PROJECT}.ods"
bq = bigquery.Client(project=PROJECT, location=LOCATION)


def sql(query: str) -> pd.DataFrame:
    """Run a query against ODS in dev; print what it scanned and how long it took."""
    config = bigquery.QueryJobConfig(default_dataset=ODS, use_query_cache=False)
    job = bq.query(query, job_config=config)
    rows = job.to_dataframe(create_bqstorage_client=False)
    seconds = (job.ended - job.started).total_seconds()
    print(f"[{(job.total_bytes_processed or 0) / 1e6:.1f} MB scanned, {seconds:.0f} s in BigQuery]")
    return rows


def dry_run_mb(query: str) -> float:
    """What a query would scan, without running it."""
    config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    return bq.query(query, job_config=config).total_bytes_processed / 1e6


ROW = "TO_JSON_STRING(JSON_STRIP_NULLS(TO_JSON(t)))"  # a row as JSON, its nulls left out
FP = f"SUM(CAST(FARM_FINGERPRINT({ROW}) AS BIGNUMERIC))"
print(f"ODS: {ODS}")

ODS: jepx-tokyo-forecast-dev.ods


## 1. What the loader records, and whether ODS agrees with it

**Step.** The loader writes a manifest row per file and ODS table, `raw_files`, whatever happened to the file ([parse README](../jtf/src/jtf/parse/README.md)). Its columns show what is recorded. BigQuery keeps metadata for every partition of a table: its row count and when it last changed (`INFORMATION_SCHEMA.PARTITIONS`), free to read. ODS tables are partitioned by `extraction_date`, so each partition holds the files of one extraction day. Per ODS table, each partition's rows are compared with the rows the manifest says were loaded into it, and its last change with the last load into it.

In [2]:
print("manifest columns:", ", ".join(c.name for c in MANIFEST_COLUMNS))

agreement = sql("""
WITH loads AS (
  SELECT ods_table, extraction_date, SUM(row_count) AS manifest_rows, MAX(ingested_at) AS last_load,
    COUNT(*) AS entries, COUNT(DISTINCT CONCAT(path, '|', parser_version)) AS files
  FROM raw_files WHERE status = 'parsed'
  GROUP BY 1, 2
),
partitions AS (
  SELECT table_name, PARSE_DATE('%Y%m%d', partition_id) AS extraction_date, total_rows,
    last_modified_time
  FROM INFORMATION_SCHEMA.PARTITIONS
  WHERE table_name != 'raw_files' AND partition_id NOT IN ('__NULL__', '__UNPARTITIONED__')
)
SELECT p.table_name, COUNT(*) AS partitions,
  COUNTIF(p.total_rows = l.manifest_rows) AS rows_as_manifest,
  COUNTIF(l.entries != l.files) AS duplicate_entries,
  MIN(TIMESTAMP_DIFF(p.last_modified_time, l.last_load, SECOND)) AS min_s_after_load,
  MAX(TIMESTAMP_DIFF(p.last_modified_time, l.last_load, SECOND)) AS max_s_after_load
FROM partitions AS p
LEFT JOIN loads AS l ON l.ods_table = p.table_name AND l.extraction_date = p.extraction_date
GROUP BY 1 ORDER BY 1
""")
agreement

manifest columns: path, source, ods_table, extraction_date, filename, fetched_at, load_type, http_status, gap, size_bytes, sha256, last_modified, status, row_count, check_result, note, parser_version, ingested_at, batch_id


[10.7 MB scanned, 2 s in BigQuery]


,table_name,partitions,rows_as_manifest,duplicate_entries,min_s_after_load,max_s_after_load
0,jepx_spot,4,4,0,3,10
1,meti_fit_capacity_city,1,1,0,59,59
2,meti_fit_capacity_pref,1,1,0,10,10
3,meti_fit_notices,3,3,0,4,9
4,openmeteo_live,4,4,0,2,4
5,openmeteo_previous_runs,2,2,0,3,22
6,openmeteo_single_runs,2,2,0,5,151
7,tepco_actuals,4,4,0,3,11
8,tepco_actuals_yearly,1,1,0,5,5
9,tepco_area_actuals_today,4,4,0,2,4


**What it shows.** **Observed:**

- **In all 17 tables, every partition holds exactly the rows the manifest says were loaded into it,** and no file is listed twice
- **A partition last changed 2 to 151 seconds after the last load into it.** `ingested_at` is stamped as a parse starts, so the gap is the parse and its load; the longest is Single Runs' history
- **The manifest records the raw file** (its `sha256` and size) **and how many rows it gave, but nothing of the rows themselves**

**Learned → design:** free metadata already shows what a load put in each partition, by count and by the time of the last change. A value changed in place keeps the count, so seeing it needs a fingerprint of the rows.

## 2. A fingerprint per file

**Step.** Three properties are wanted. The same rows always give the same fingerprint, whatever order BigQuery reads them in. Any change gives a different one, a row loaded twice included. And a column added later, null in the old rows, leaves old fingerprints alone. The candidate hashes each row's JSON with `FARM_FINGERPRINT` (**Source**: [BigQuery hash functions](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/hash_functions)), drops nulls first with `JSON_STRIP_NULLS` (**Source**: [BigQuery JSON functions](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/json_functions)), and sums the hashes as `BIGNUMERIC`, exact arithmetic, so the sum is the same in any order (step 5 learned that BigQuery's float `SUM` isn't). The common alternative, `BIT_XOR` of the hashes, is compared on a file loaded twice. The key is a file and its parser version, as in the manifest: a re-parse appends its rows under a new version, with a fingerprint of its own.

**2.1** Every file of `tepco_actuals`, fingerprinted twice by queries of different shapes.

In [3]:
by_group = sql(f"""
SELECT source_file, parser_version, COUNT(*) AS row_count, {FP} AS fingerprint
FROM tepco_actuals AS t GROUP BY 1, 2
""")
by_window = sql(f"""
SELECT DISTINCT source_file, parser_version,
  COUNT(*) OVER file AS row_count, {FP} OVER file AS fingerprint
FROM tepco_actuals AS t
WINDOW file AS (PARTITION BY source_file, parser_version)
""")
both = by_group.merge(by_window, on=["source_file", "parser_version"], suffixes=("", "_again"))
same = (both.fingerprint == both.fingerprint_again).all()
print(f"files: {len(by_group)} | the same fingerprint both ways: {same}")

[19.1 MB scanned, 1 s in BigQuery]


[19.1 MB scanned, 1 s in BigQuery]
files: 46 | the same fingerprint both ways: True


**What it shows.** **Observed:** all 46 files of `tepco_actuals` give the same fingerprint from both query shapes.

**2.2** One file loaded twice, against the same file once, under the sum and under `BIT_XOR`.

In [4]:
FILE = by_group.sort_values("row_count").source_file.iloc[len(by_group) // 2]
COPY = ROW.replace(
    "TO_JSON(t)", "TO_JSON((SELECT AS STRUCT t.* EXCEPT (copies)))"
)  # without `copies`
twice = sql(f"""
WITH once AS (SELECT * FROM tepco_actuals WHERE source_file = '{FILE}'),
copies AS (SELECT 1 AS copies, t.* FROM once AS t UNION ALL
           SELECT 2, t.* FROM once AS t UNION ALL SELECT 2, t.* FROM once AS t)
SELECT copies, COUNT(*) AS row_count,
  SUM(CAST(FARM_FINGERPRINT({COPY}) AS BIGNUMERIC)) AS sum_fp,
  BIT_XOR(FARM_FINGERPRINT({COPY})) AS xor_fp
FROM copies AS t GROUP BY copies ORDER BY copies
""")
print(f"file: {FILE}")
twice

[19.1 MB scanned, 0 s in BigQuery]
file: tepco_actuals/2026-11-15/20261115T045421Z_eria_jukyu_202609_03.csv


,copies,row_count,sum_fp,xor_fp
0,1,1440,226157147187861571564.00000000000000000000000000000000000000,4877300710771716426
1,2,2880,452314294375723143128.00000000000000000000000000000000000000,0


**What it shows.** **Observed:** loaded twice, a file's sum doubles. Under `BIT_XOR` its hashes cancel to exactly 0, as if the file had no rows.

**Learned → design:** the sum, not XOR. A duplicate load is the one way ODS is known to go wrong today (two Dagsters at once), and it must show.

**2.3** What a fingerprint costs: one file, read with its partition and file given as constants, against each whole ODS table.

In [5]:
def file_mb(path: str) -> float:
    """A file's fingerprint, read with its partition and file as constants."""
    day = path.split("/")[1]
    query = f"SELECT {FP} FROM `{ODS}.tepco_actuals` AS t"
    return dry_run_mb(f"{query} WHERE extraction_date = '{day}' AND source_file = '{path}'")


daily_file = by_group[by_group.source_file.str.contains("2026-11-17/")].source_file.iloc[0]
history_file = by_group[by_group.source_file.str.contains("2026-11-15/")].source_file.iloc[0]
tables = sql("SELECT table_id FROM __TABLES__ WHERE table_id != 'raw_files'").table_id
whole = pd.Series(
    {t: dry_run_mb(f"SELECT {FP} FROM `{ODS}.{t}` AS t") for t in tables}
).sort_values(ascending=False)
print(f"a daily file: {file_mb(daily_file):.1f} MB")
print(f"a file of the history load: {file_mb(history_file):.1f} MB")
print(f"all of ODS: {whole.sum():,.0f} MB")
whole.head(6).round(1)

[0.0 MB scanned, 0 s in BigQuery]


a daily file: 0.7 MB
a file of the history load: 16.9 MB
all of ODS: 3,792 MB


meti_fit_capacity_city       2708.7
openmeteo_previous_runs       488.4
openmeteo_single_runs         295.3
tepco_demand_outlook_5min     111.6
meti_fit_capacity_pref         68.3
jepx_spot                      67.7
dtype: float64

**What it shows.** **Observed:** a file's fingerprint reads at most its partition: 0.7 MB for a daily file, 16.9 MB for a file of the history load, whose partition holds every history file (a dry run estimates the whole partition; clustering by file may read less). All of ODS is 3.8 GB, 71% of it the capacity source's city table.

**Learned → design:** fingerprinting each file right after its load costs about what the load does. Rechecking everything after every load would read 3.8 GB each time, several times a day; only partitions that may have changed should be read.

## 3. A change in place, in a scratch dataset

**Step.** A throwaway dataset gets a copy of `tepco_actuals` holding three files on three extraction days. Each is loaded the way the loader loads, through `jtf`'s `BigQueryWarehouse.append`, and fingerprinted right after its load: the fingerprint recorded at load. Then four changes, one at a time, each followed by a look at the partitions' metadata and at every file's fingerprint:

1. An `UPDATE` of one value in file A
2. A `DELETE` of one row of file B
3. File C loaded a second time, as two Dagsters at once would (progress, Known issues)
4. A new nullable column, the way a schema change would add one

The dataset is dropped at the end.

In [6]:
SCRATCH = f"{PROJECT}.scratch_fingerprint"
dataset = bigquery.Dataset(SCRATCH)
dataset.location = LOCATION
bq.create_dataset(dataset, exists_ok=True)
scratch = BigQueryWarehouse(PROJECT, "scratch_fingerprint", client=bq)
parser = PARSERS["tepco_actuals"]
TABLE = f"{SCRATCH}.{parser.table}"

daily = by_group[by_group.source_file.str.contains("_202611_03")].sort_values("source_file")
picked = daily.groupby(daily.source_file.str.split("/").str[1]).head(1).tail(3).source_file.tolist()
FILES = dict(zip("ABC", picked, strict=True))
recorded, loaded_at = {}, {}
for name, path in FILES.items():
    rows = bq.query(
        f"SELECT * FROM `{ODS}.tepco_actuals` WHERE source_file = '{path}'"
    ).to_dataframe(create_bqstorage_client=False)
    scratch.append(parser.table, parser.schema, "source_file", rows)
    loaded_at[name] = datetime.now(UTC)
    recorded[name] = (
        bq.query(f"SELECT {FP} AS fp FROM `{TABLE}` AS t WHERE source_file = '{path}'")
        .to_dataframe(create_bqstorage_client=False)
        .fp.iloc[0]
    )


def look(step: str) -> pd.DataFrame:
    """Each file's partition metadata, and whether its rows still match its fingerprint."""
    parts = bq.query(f"""
      SELECT PARSE_DATE('%Y%m%d', partition_id) AS day, total_rows, last_modified_time
      FROM `{SCRATCH}.INFORMATION_SCHEMA.PARTITIONS` WHERE table_name = '{parser.table}'
    """).to_dataframe(create_bqstorage_client=False)
    now = bq.query(f"""
      SELECT source_file, extraction_date AS day, COUNT(*) AS row_count, {FP} AS fp
      FROM `{TABLE}` AS t GROUP BY 1, 2
    """).to_dataframe(create_bqstorage_client=False)
    rows = []
    for name, path in FILES.items():
        file = now[now.source_file == path].iloc[0]
        part = parts[parts.day == file.day].iloc[0]
        rows.append(
            {
                "step": step,
                "file": name,
                "rows": file.row_count,
                "partition_rows": part.total_rows,
                "modified": part.last_modified_time,
                "as_loaded": file.fp == recorded[name],
            }
        )
    return pd.DataFrame(rows)


steps = [look("loaded")]
print("files:", FILES)
print("loads done at:", {k: f"{v:%H:%M:%S}" for k, v in loaded_at.items()})
steps[0]

files: {'A': 'tepco_actuals/2026-11-16/20261115T234520Z_eria_jukyu_202611_03.csv', 'B': 'tepco_actuals/2026-11-17/20261116T234521Z_eria_jukyu_202611_03.csv', 'C': 'tepco_actuals/2026-11-18/20261117T234520Z_eria_jukyu_202611_03.csv'}
loads done at: {'A': '09:28:29', 'B': '09:28:33', 'C': '09:28:35'}


,step,file,rows,partition_rows,modified,as_loaded
0,loaded,A,737,737,2026-11-18 09:28:29.212+00:00,True
1,loaded,B,785,785,2026-11-18 09:28:31.922+00:00,True
2,loaded,C,833,833,2026-11-18 09:28:35.464+00:00,True


**What it shows.** **Observed:** each partition's last change falls one to two seconds before the loader's call returns: at the end of its load job.

In [7]:
def run(statement: str) -> None:
    bq.query(statement).result()


run(f"UPDATE `{TABLE}` SET solar = solar + 1 WHERE source_file = '{FILES['A']}' AND row_number = 1")
steps.append(look("1. UPDATE in A"))
run(f"DELETE FROM `{TABLE}` WHERE source_file = '{FILES['B']}' AND row_number = 1")
steps.append(look("2. DELETE in B"))
again = bq.query(
    f"SELECT * FROM `{ODS}.tepco_actuals` WHERE source_file = '{FILES['C']}'"
).to_dataframe(create_bqstorage_client=False)
scratch.append(parser.table, parser.schema, "source_file", again)
steps.append(look("3. C loaded twice"))


def whole_table(fingerprint: str) -> object:
    query = f"SELECT {fingerprint} AS fp FROM `{TABLE}` AS t"
    return bq.query(query).to_dataframe(create_bqstorage_client=False).fp.iloc[0]


KEPT = "SUM(CAST(FARM_FINGERPRINT(TO_JSON_STRING(t)) AS BIGNUMERIC))"  # the nulls kept in
before = {"without nulls": whole_table(FP), "nulls kept": whole_table(KEPT)}
run(f"ALTER TABLE `{TABLE}` ADD COLUMN added_later STRING")
steps.append(look("4. a column added"))
after = {"without nulls": whole_table(FP), "nulls kept": whole_table(KEPT)}

history = pd.concat(steps, ignore_index=True)
history["moved"] = history.groupby("file").modified.diff().dt.total_seconds().gt(0)
print(
    "a new column changes the table's fingerprint:",
    {k: bool(before[k] != after[k]) for k in before},
)
history.drop(columns="modified")

a new column changes the table's fingerprint: {'without nulls': False, 'nulls kept': True}


,step,file,rows,partition_rows,as_loaded,moved
0,loaded,A,737,737,True,False
1,loaded,B,785,785,True,False
2,loaded,C,833,833,True,False
3,1. UPDATE in A,A,737,737,False,True
4,1. UPDATE in A,B,785,785,True,False
5,1. UPDATE in A,C,833,833,True,False
6,2. DELETE in B,A,737,737,False,False
7,2. DELETE in B,B,784,784,False,True
8,2. DELETE in B,C,833,833,True,False
9,3. C loaded twice,A,737,737,False,False


**What it shows.** **Observed:**

- **The `UPDATE` moved only A's partition,** and only A's fingerprint changed. Its rows stayed 737, so a count alone would miss it
- **The `DELETE` moved only B's partition,** and took it to 784 rows
- **The second load of C moved only C's partition,** and doubled its rows
- **The new column moved no partition.** Without nulls the table's fingerprint stayed as it was; with nulls kept, it changed

BigQuery's own description adds a caveat this run didn't show: `last_modified_time` is updated when data is written, but *"Modifications that involve record deletions might not be reflected"* (**Source**: [INFORMATION_SCHEMA.PARTITIONS](https://docs.cloud.google.com/bigquery/docs/information-schema-partitions)). A delete changes the count, though.

**Learned → design:** the two pieces of metadata together point at every partition a change touched, and at nothing else seen here. So after each load, only partitions whose rows differ from what was recorded, or whose last change came after the fingerprint recorded for their last load, need their rows read: normally none. Each fingerprint is recorded after its load job ends, so a later change shows a `last_modified_time` after it, with no tolerance to guess.

In [8]:
bq.delete_dataset(SCRATCH, delete_contents=True, not_found_ok=True)
print("scratch dataset dropped:", SCRATCH)

scratch dataset dropped: jepx-tokyo-forecast-dev.scratch_fingerprint


## The design this leads to

| Piece | What | Where |
| --- | --- | --- |
| The fingerprint | Per file and parser version: the sum, as `BIGNUMERIC`, of `FARM_FINGERPRINT` over each row as JSON without its nulls | One SQL expression, shared by the loader and the check |
| Recorded at load | Right after a parse appends its rows, BigQuery fingerprints each loaded file with its partition and file as constants, and the loader appends one row per file to an append-only table beside the manifest: table, file, version, extraction day, row count, fingerprint, when recorded, and how (`load`). A file the parse finds already loaded but not in the manifest (a stopped load) gets its fingerprint then, as it gets its manifest row | `jtf.parse` |
| The files already in ODS | Fingerprinted once, recorded the same way with `backfill`: guarded from then on, not from their load | A one-off command |
| The check, after every load | Per ODS table, from free metadata: partitions whose rows differ from the recorded counts, or whose last change came after the last fingerprint recorded for them. Only their files are fingerprinted again, with constants, and compared. Any file whose rows differ, that was loaded twice, or that has no fingerprint fails the check by name | An asset check on each ODS asset, beside `files_pass_checks` |
| A full recheck | Every file, 3.8 GB: before step 6's backfill-stability test, and on demand | A `jtf` command |

## What this exploration made known

| We assumed | We found | So |
| --- | --- | --- |
| The manifest is a record of what ODS holds | It records the raw file and the count of its rows, nothing of the rows (section 1) | A fingerprint per file, recorded at its load |
| Any hash of the rows would do | `BIT_XOR` cancels a file loaded twice to 0; nulls kept in make a new column change every old row's fingerprint (sections 2.2, 3) | The sum of `FARM_FINGERPRINT` as `BIGNUMERIC`, over each row as JSON without its nulls |
| A recheck after every load reads ODS | All of ODS is 3.8 GB; one file reads at most its partition (section 2.3) | Read only the partitions that may have changed |
| Something has to say which partitions changed | A partition's metadata moved exactly where an `UPDATE`, a `DELETE` or a load touched it, and not for a new column. ODS agrees with the manifest in every partition today (sections 1, 3). BigQuery warns that deletions might not move `last_modified_time` | The check screens partitions by both `total_rows` and `last_modified_time`, at no cost, and rereads only those flagged |
| The plan's check is a dbt data test against `raw_files` | Choosing partitions takes a lookup before the reading query, since BigQuery prunes partitions only by constants ([notebook 04](04_ods_to_dwd.ipynb), section 6.8); the rows are ODS, which `jtf` loads | An asset check on each ODS asset, after its load |

### Still open

- **The manifest and the fingerprint table are not fingerprinted.** Their own partition metadata still shows a change after their last append, but nothing records what their rows were. A record of a record would only move the question
- **Automatic re-clustering** has not moved a partition's last change so far. If it ever does, the check rereads that partition and finds the rows unchanged: the cost is bytes, not a false failure
- **Restoring a changed partition:** BigQuery's time travel keeps seven days of history, but there is no runbook for using it

## Where the sources and the data disagree

| Where | Says | We found | Handled by |
| --- | --- | --- | --- |
| [Phase-1 plan](../docs/phase1/README.md#ingestion-and-warehouse), Warehouse tests; [rules README](../warehouse/dbt/rules/README.md) | The fingerprint is checked by "a dbt data test against `ods.raw_files`, after every load" | The check has to choose its partitions before it reads, and it guards what `jtf` loads (section 3) | An asset check on each ODS asset; the plan and the rules README to be updated |
| The same | The fingerprint "changes the loader and the manifest" | Files already in ODS get a fingerprint later than their load, which is not a parse the manifest could record | A table of fingerprints beside the manifest, append-only, each row saying whether it was recorded at `load` or by `backfill` |

## Sources

| Page | Supports |
| --- | --- |
| [BigQuery `INFORMATION_SCHEMA.PARTITIONS`](https://docs.cloud.google.com/bigquery/docs/information-schema-partitions) | `last_modified_time`, *"The most recent time that data was written to the partition"*, *"updated when data is inserted, loaded, streamed, or modified within the partition"*, and *"Modifications that involve record deletions might not be reflected"*; `total_rows`, *"The total number of rows in the partition"* |
| [BigQuery hash functions](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/hash_functions) | `FARM_FINGERPRINT`: *"The output of this function for a particular input never changes across runs or query engine implementations"* |
| [BigQuery JSON functions](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/json_functions) | `TO_JSON` of a row, and `JSON_STRIP_NULLS` |
| [Notebook 04](04_ods_to_dwd.ipynb), section 6.8 | Partitions are pruned only by constants |
| [parse README](../jtf/src/jtf/parse/README.md) | The manifest, and a parse once per version |